In [1]:
#Gen imports

import numpy as np
from matplotlib import pyplot as plt
import nibabel as nb
from scipy.ndimage import gaussian_filter
from scipy.ndimage import distance_transform_edt
from glob import glob
import tifffile as tiff
import sys
import os
from glob import glob
sys.path.append('../')
from scipy import ndimage as ndi
from skimage.measure import block_reduce
from pathlib import Path
import pandas as pd
from scipy.optimize import linear_sum_assignment


import re

# from slice_structure_identification_functions import compute_signed_distance_weight as compute_signed_distance_weight
# from slice_structure_identification_functions import compute_signed_distance_weight_filled as compute_signed_distance_weight_filled

import importlib


import slice_registration_functions
importlib.reload(slice_registration_functions)
from slice_registration_functions import apply_coordinate_mapping_2d, downsample_image, build_centroid_image



nighres not found, skipping
nighres not found, skipping
nighres not found, skipping


In [2]:
mask_dir     = "/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/filtered_binary_masks"
centroid_csv = "/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/filtered_binary_masks/centroids.csv"
reg_dir      = "/data2/neuralabc/johmat/zefir_sliceReg_optimized_v2_rescale_5"
out_dir      = "/data/neuralabc/johmat/phase_ml/processing/full_slide_masks/registered_centroids"

mask_res = 0.3449 # um per pixel in the original mask images
invmap_res = 50 #base res is 10um, mult by the rescale factor in the dir name

rescale  = int(invmap_res / mask_res)       
# rescale = 145          
PROP_PAD = 0.2                # the pipeline default. confirm with the test overlay below

fname_suffixes = [
    "coreg0nl_ants",
    "coreg12nl_win12_rigsyn_4_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter3_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter4_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter5_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter6_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter7_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter8_ants",
    "coreg12nl_win12_rigsyn_4_groupwise_iter9_ants",
]

In [ ]:
## generate and save the downsampled label count images for each slice, then apply the mapping chain to them and compare to the final def0 image. This is a sanity check to make sure the mapping chain is working correctly.

# orig_fnames = sorted(glob(os.path.join(root_dir, 'zefir_????_*_pix.nii.gz')))
# #hardcode specific slice with large visual shift for testing
orig_fnames = sorted(glob(os.path.join(root_dir, '*realID_*_zefirID_????_*_pix.nii.gz')))

max_errors = []
missing_slices = []
bad_slices = [] #exceed max_error_thresh
max_error_thresh = 5.0
final_space_labels = {}
final_space_labels['img'] =[]
final_space_labels['fname_header'] =[]
final_space_labels['final_reg_space_def0'] = []
for name_idx, orig_fname in enumerate(orig_fnames):

    slice_name = os.path.basename(orig_fname)
    base_slice = slice_name.split('_downsample_10p002um_pix')[0]

    # Extract the shared prefix token even when filenames have leading tags.
    header_match = re.search(r'(realID_[\d-]+_zefirID_\d+)', base_slice)
    if header_match is None:
        continue
    fname_header = header_match.group(1)
    labeled_mask = glob(os.path.join(mask_dir, f"*{fname_header}_*_mask*.ome.tif"))

    if len(labeled_mask) == 0:
        # print(f"No labeled mask found for {slice_name}")
        continue
    elif len(labeled_mask) > 1:
        # print(f"Multiple labeled masks found for {slice_name}: {labeled_mask}")
        continue

    labeled_mask_fname = labeled_mask[0]
    if os.path.exists(labeled_mask_fname):
        print(f"Found labeled mask for {slice_name}: {labeled_mask_fname}")

    source_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix.nii.gz")
    mapping_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg0nl_ants-map.nii.gz")
    mapping_img3 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_ants-map.nii.gz")
    mapping_img4 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter3_ants-map.nii.gz")
    mapping_img5 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter4_ants-map.nii.gz")
    mapping_img6 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter5_ants-map.nii.gz")
    mapping_img7 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter6_ants-map.nii.gz")
    mapping_img8 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter7_ants-map.nii.gz")
    mapping_img9 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter8_ants-map.nii.gz")
    mapping_img10 = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-map.nii.gz")

    final_comparison_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter3_ants-def0.nii.gz")
    final_comparison_img = os.path.join(root_dir, f"{base_slice}_downsample_10p002um_pix_coreg12nl_win12_rigsyn_4_groupwise_iter9_ants-def0.nii.gz")

    if not (os.path.exists(source_img) and os.path.exists(mapping_img) and os.path.exists(mapping_img3) and os.path.exists(mapping_img4)):
        missing_slices.append(slice_name)
        continue
    print(slice_name)
    src = nb.load(source_img)
    m1_ = nb.load(mapping_img)
    m2_ = nb.load(mapping_img3)
    m3_ = nb.load(mapping_img4)
    m4_ = nb.load(mapping_img5)
    m5_ = nb.load(mapping_img6)
    m6_ = nb.load(mapping_img7)
    m7_ = nb.load(mapping_img8)
    m8_ = nb.load(mapping_img9)
    m9_ = nb.load(mapping_img10)

    labeled_mask = tiff.imread(labeled_mask_fname).astype(bool).astype(np.uint8) # read and convert to bin

    # centroid_seed_img = build_centroid_image(labeled_mask)
    centroid_seed_img = labeled_mask #skipping centroids bc it takes too long for testing
    _ds_label_cnt = downsample_image(centroid_seed_img, rescale, prop_pad=0)
    _ds_label_cnt = center_crop_or_pad_2d(_ds_label_cnt, target_shape=src.shape)
    # _ds_label_cnt = downsample_image(labeled_mask, rescale, pad_value=-1*0) #now counts per pixel

    ## this breaks because the downsampled label count image is not the same shape as the source image
    # _d = np.zeros_like(src.get_fdata())
    # _d[...] = _ds_label_cnt
    # label_cnt_img = nb.Nifti1Image(_d, affine=src.affine, header=src.header)

    label_cnt_img = nb.Nifti1Image(_ds_label_cnt, affine=src.affine, header=src.header)



    # if src.shape != label_cnt_img.shape:
    #     raise ValueError(f"Shape mismatch betweem source image and label count image: {src.shape} vs {label_cnt_img.shape}")

    # #create sham data full of 0s with one value in the middle of the image to track how it moves across the transformations
    # sham_data = np.zeros(src.shape, dtype=np.float32)
    # label_cnt_img = nb.Nifti1Image(_ds_label_cnt, affine=src.affine, header=src.header)

    comparison = nb.load(final_comparison_img).get_fdata()
    # seq = apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(apply_coordinate_mapping_2d(src, m1_), m2_), m3_).get_fdata()
    labeled_seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        label_cnt_img, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_)

    seq = apply_coordinate_mapping_2d(
        apply_coordinate_mapping_2d(
            apply_coordinate_mapping_2d(
                apply_coordinate_mapping_2d(
                    apply_coordinate_mapping_2d(
                        apply_coordinate_mapping_2d(
                            apply_coordinate_mapping_2d(
                                apply_coordinate_mapping_2d(
                                    apply_coordinate_mapping_2d(
                                        src, m1_), m2_), m3_), m4_), m5_), m6_), m7_), m8_), m9_).get_fdata()

    final_space_labels['img'].append(labeled_seq)
    final_space_labels['fname_header'].append(fname_header)
    final_space_labels['final_reg_space_def0'].append(nb.load(final_comparison_img))

    max_err = np.abs(seq -comparison).max()
    max_errors.append(max_err)
    if max_err > max_error_thresh:
        bad_slices.append((slice_name, max_err, name_idx))

max_errors = np.array(max_errors)
print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

print(f"Processed {len(max_errors)} slices successfully.")
if missing_slices:
    print(f"Skipped {len(missing_slices)} slices due to missing files: {missing_slices[:10]}{'...' if len(missing_slices) > 10 else ''}")

plt.figure(figsize=(10, 5))
plt.hist(max_errors, bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison)')
plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()

plt.figure(figsize=(10, 5))
plt.hist(max_errors[max_errors<max_error_thresh], bins=20, color='tab:blue', edgecolor='black')
plt.title('Max error per slice: apply_mapping_chain_exact vs def0 (final comparison, missing slices removed)')

plt.xlabel('Max error')
plt.ylabel('Number of slices')
plt.grid(True, linestyle='--', alpha=0.4)
plt.show()


